In [1]:

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd

load_dotenv(find_dotenv())

USABILITY_THRESHOLD = 0.85  # threshold of data availability to be considered usable

STATUSES = ['Active', 'Final', 'Inactive', 'In Review']
DATE_CONCEPTS = ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 'PERMIT_OR_FILE_DATE']

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

REPORT_FILEPATH = os.path.join(ROOT_PATH, "reports", "2026-07-22-data-usability-report-top-50.md")

SUMMARY_FILENAME = "dewey_summary"
SUMMARY_FILEPATH = os.path.join(MY_DATA_PATH, f"{SUMMARY_FILENAME}.parquet")
TOP_50_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data", "top_50_us_cities.csv")

COLUMNS = [
    'JURISDICTION', 'STATE', 'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 'STATUS_NORMALIZED', 'RECORD_TYPE_ORIGINAL', 'APN', 'STREET', 'ZIPCODE'
]  # Columns to load from data file

MD = """
# Building Permits Data Usability Report

For the top 50 US cities by population. 

Key variables:
- FILE_DATE: Application date of permit
- PERMIT_DATE: Issue date of permit
- FINAL_DATE: Final date of permit
- PERMIT_OR_FILE_DATE: PERMIT_DATE if available, FILE_DATE otherwise
- STATUS_NORMALIZED: Normalized status of permit: "Active", "Final", "Inactive", "In Review"

Notes:
- This report checks the availability of the key date fields by the status of the permit and reports on the number of cities with usable data.
- This report does not de-duplicate the data. It is known that there are permit duplicates, but we have not yet investigated the extent of the problem.
- This report matches each city to a single jurisdiction in the permits data. It is possible that one city or metro area could be served by multiple jurisdictions (a jurisdiction is a building permitting authority). Thus, in real work, we'd likely want to match permits to cities based on the geocoded addresses, not on the name of the jurisdiction.
- Nuances regarding the correct interpretation of FILE_DATE, PERMIT_DATE, and FINAL_DATE may depend on the jurisdiction and warrants further investigation.
- Reported date ranges are based on 1st and 99th percentiles to avoid outliers (which are likely recording errors).

"""

In [2]:
# Load the data

dfs = pd.read_parquet(SUMMARY_FILEPATH)
dfc = pd.read_csv(TOP_50_FILEPATH)

dfc = dfc.rename(columns={'CITY': 'JURISDICTION'})

In [3]:
# Notes:
# - In Dewey, the closest match for Las Vegas is "North Las Vegas"
# - In Dewey, the closest match for Louisville is "Louisville-Jefferson County"
# - In Dewey, the closest match for Colorado Springs is "El Paso County"
# - In Dewey, there does not appear to be a match for Wichita, KS

JURISDICTION_MAP = {}
for idx, row in dfc.iterrows():
    JURISDICTION_MAP[row['JURISDICTION']] = row['JURISDICTION']

JURISDICTION_MAP['Las Vegas'] = 'North Las Vegas'
JURISDICTION_MAP['Louisville'] = 'Louisville-Jefferson County'
JURISDICTION_MAP['Colorado Springs'] = 'El Paso County'

In [4]:
# Iterate over cities

USABILITY_STATS = {}

t0 = time.time()
for idx, row in dfc.iterrows():
    j = row['JURISDICTION']
    jurisdiction = JURISDICTION_MAP[j]
    state = row['STATE']
    dt = time.time() - t0

    if (j, state) not in USABILITY_STATS:
        USABILITY_STATS[(j, state)] = {}

    print(f"\rProcessing {j} {state} ({idx + 1}/{len(dfc)}) ... elapsed time {dt:.2f} seconds                ", end="", flush=True)

    MD += f"\n## {j}, {state}\n\n"

    if jurisdiction != j:
        MD += f"*Note: The best match for {j}, {state} in the permits data was {jurisdiction}, {state}*.\n\n"

    files = dfs.loc[(dfs['JURISDICTION'] == jurisdiction) & (dfs['STATE'] == state), 'FILENAME'].tolist()
    mydf = []
    for f in files:
        temp_df = pd.read_parquet(os.path.join(DEWEY_PATH, f), columns=COLUMNS)
        temp_df = temp_df.loc[(temp_df['JURISDICTION'] == jurisdiction) & (temp_df['STATE'] == state)]
        mydf.append(temp_df)

    if len(mydf)==0:
        MD += f"**No permits data found for {jurisdiction}, {state}**.\n\n"
        continue

    mydf = pd.concat(mydf)

    # Clean date fields
    mydf['FILE_DATE'] = pd.to_datetime(mydf['FILE_DATE'], errors='coerce')
    mydf['PERMIT_DATE'] = pd.to_datetime(mydf['PERMIT_DATE'], errors='coerce')
    mydf['FINAL_DATE'] = pd.to_datetime(mydf['FINAL_DATE'], errors='coerce')
    mydf['PERMIT_OR_FILE_DATE'] = mydf['PERMIT_DATE'].fillna(mydf['FILE_DATE'])

    # Total permits
    total_permits = len(mydf)
    MD += f"- Total permits: {total_permits:,}\n"

    # STATUS_NORMALIZED
    status_normalized_not_na = mydf['STATUS_NORMALIZED'].notna().sum()
    status_normalized_pct = status_normalized_not_na/(total_permits + 1e-6)
    status_normalized_usable = "*OK*" if status_normalized_pct >= USABILITY_THRESHOLD else "**FAIL**"
    MD += f"- STATUS_NORMALIZED not missing: {status_normalized_not_na:,} ({status_normalized_not_na/total_permits:.1%}) - {status_normalized_usable}\n"

    for status in STATUSES:
        if status not in USABILITY_STATS[(j, state)]:
            USABILITY_STATS[(j, state)][status] = {}

        n_status = (mydf['STATUS_NORMALIZED']==status).sum()
        pct_status = n_status / (status_normalized_not_na + 1e-6)
        MD += f"    - {status}: {n_status:,} ({pct_status:.1%})\n"
        for dc in DATE_CONCEPTS:
            if dc not in USABILITY_STATS[(j, state)][status]:
                USABILITY_STATS[(j, state)][status][dc] = False

            mask = (mydf['STATUS_NORMALIZED']==status) & (mydf[f'{dc}'].notna())
            n_dc = mask.sum()
            pct_dc = n_dc / (n_status + 1e-6)
            dc_usable = "*OK*" if pct_dc >= USABILITY_THRESHOLD else "**FAIL**"
            USABILITY_STATS[(j, state)][status][dc] = (pct_dc >= USABILITY_THRESHOLD)
            year_p1 = mydf.loc[mask, f'{dc}'].quantile(0.01).year
            year_p99 = mydf.loc[mask, f'{dc}'].quantile(0.99).year
            MD += f"        - {dc} non-missing: {n_dc:,} ({pct_dc:.1%}) [{year_p1}-{year_p99}]- {dc_usable} \n"


Processing Arlington TX (50/50) ... elapsed time 67.18 seconds                       

In [5]:
# Number of cities with usable data under different data needs
MD += "\n\n## By data requirements\n\n"

MD += "- Require FILE_DATE for all statuses, PERMIT_DATE for all but 'In Review', FINAL_DATE for 'Final': "
n_usable = 0
for k, v in USABILITY_STATS.items():
    j, state = k
    usable = True
    if 'Active' not in v:
        usable = False
        continue
    for status in ['Active', 'Inactive', 'In Review', 'Final']:
        if not v[status]['FILE_DATE']:
            usable = False
            break
    for status in ['Active', 'Inactive', 'Final']:

        if not v[status]['PERMIT_DATE']:
            usable = False
            break
    for status in ['Final']:
        if not v[status]['FINAL_DATE']:
            usable = False
            break
    if usable:
        n_usable += 1
MD += f'{n_usable} / {len(dfc)} \n'

MD += "- Require FILE_OR_PERMIT_DATE for all statuses, and FINAL_DATE for 'Final': "
n_usable = 0
for k, v in USABILITY_STATS.items():
    j, state = k
    usable = True
    if 'Active' not in v:
        usable = False
        continue
    for status in ['Active', 'Inactive', 'In Review', 'Final']:
        if not v[status]['PERMIT_OR_FILE_DATE']:
            usable = False
            break
    for status in ['Final']:
        if not v[status]['FINAL_DATE']:
            usable = False
            break
    if usable:
        n_usable += 1
MD += f'{n_usable} / {len(dfc)} \n'

MD += "- Require FILE_OR_PERMIT_DATE for all statuses: "
n_usable = 0
for k, v in USABILITY_STATS.items():
    j, state = k
    usable = True
    if 'Active' not in v:
        usable = False
        continue
    for status in ['Active', 'Inactive', 'In Review', 'Final']:
        if not v[status]['PERMIT_OR_FILE_DATE']:
            usable = False
            break
    if usable:
        n_usable += 1
MD += f'{n_usable} / {len(dfc)} \n'

MD += "\n"



In [6]:
CONCLUSION = """

## Conclusion

- PERMIT_OR_FILE_DATE appears mostly usable. 
- FINAL_DATE appears usable only for some jurisdictions. 
- Need to investigate:
    - If and when PERMIT_DATE and FILE_DATE are interchangeable
    - If and when FINAL_DATE is available
    - Why certain dates are not available for certain jurisdictions

"""

with open(REPORT_FILEPATH, 'w') as f:
    f.write(MD + CONCLUSION)